# Recallia Quest — fine-tune the private remembrance model

This notebook trains **recallia-remember**: a small model that does one job only — turn a caretaker's evidence into a grounded remembrance-story JSON — and refuses anything else.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.
When asked, upload `train.jsonl` and `val.jsonl` from the `finetune/data` folder of Recallia Quest.

Training uses only synthetic (invented) families, never real family data. Takes about 30–60 minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from google.colab import files
print('Upload train.jsonl and val.jsonl (from recallia-quest/finetune/data)')
up = files.upload()
assert 'train.jsonl' in up and 'val.jsonl' in up, 'Please upload both train.jsonl and val.jsonl'

In [ ]:
# ---- settings ----
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct'   # good balance of accuracy and speed; 'unsloth/Qwen2.5-7B-Instruct' is more accurate but slower
MAX_LEN = 4096
EPOCHS = 2

from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=3407)

In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')

def load(path):
    rows = [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]
    return rows, Dataset.from_list([{'text': tokenizer.apply_chat_template(r['messages'], tokenize=False)} for r in rows])

train_rows, train_ds = load('train.jsonl')
val_rows, val_ds = load('val.jsonl')
lens = [len(tokenizer(t)['input_ids']) for t in train_ds['text'][:300]]
print(len(train_ds), 'train /', len(val_ds), 'val  | tokens per example ~', sum(lens)//len(lens), 'max', max(lens))

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

args = SFTConfig(
    dataset_text_field='text', max_seq_length=MAX_LEN, packing=False,
    per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=EPOCHS,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03, weight_decay=0.01, optim='adamw_8bit',
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    logging_steps=10, eval_strategy='steps', eval_steps=100, save_strategy='no', seed=3407, output_dir='outputs', report_to='none')
kw = {'processing_class': tokenizer} if 'processing_class' in inspect.signature(SFTTrainer.__init__).parameters else {'tokenizer': tokenizer}
trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=args, **kw)
# Learn only the answers (the story JSON), not the long prompt.
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
stats = trainer.train()
print(stats)

## Quick accuracy check on held-out examples
Checks JSON validity, that only the given photo/audio ids are used, and that every memory/question cites real evidence. The full production validator runs on your computer with `npm run ft:eval`.

In [ ]:
import re
FastLanguageModel.for_inference(model)
def check(row, text):
    try:
        j = json.loads(text[text.index('{'): text.rindex('}') + 1])
    except Exception:
        return 'bad-json'
    if row['meta']['kind'] == 'refusal':
        return 'ok' if 'error' in j and 'steps' not in j else 'no-refusal'
    ev = row['meta']['ev']
    ids = {p['id'] for p in ev['photos']} | {a['id'] for a in ev['audio']}
    codes = {f['code'] for f in ev['facts']}
    for s in j.get('steps', []):
        if s.get('type') in ('photo', 'audio') and s.get('itemId') not in ids: return 'bad-id'
        if s.get('type') in ('memory', 'question') and not set(s.get('evidence') or []) & codes: return 'no-evidence'
    return 'ok'

results = {}
for row in val_rows[:30]:
    msgs = [m for m in row['messages'] if m['role'] != 'assistant']
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=1500, temperature=0.2, do_sample=True)
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    r = check(row, text); results[r] = results.get(r, 0) + 1
print(results)
print(text[:1500])

## Export for Ollama
Creates a quantised GGUF file (about 2 GB) and downloads it with the Modelfile. Keep the browser tab open until the download finishes.

In [ ]:
import glob, os
model.save_pretrained_gguf('recallia-remember', tokenizer, quantization_method='q4_k_m')
gguf = sorted(glob.glob('recallia-remember*/*.gguf') + glob.glob('*.gguf'), key=os.path.getsize)[-1]
os.rename(gguf, 'recallia-remember.Q4_K_M.gguf')
print('ready:', os.path.getsize('recallia-remember.Q4_K_M.gguf') // 2**20, 'MB')

In [ ]:
# Optional: save a copy to Google Drive instead of (or as well as) downloading
# from google.colab import drive; drive.mount('/content/drive'); !cp recallia-remember.Q4_K_M.gguf /content/drive/MyDrive/
files.download('recallia-remember.Q4_K_M.gguf')

## On your computer
1. Move the downloaded `recallia-remember.Q4_K_M.gguf` into the `recallia-quest/finetune/` folder.
2. Keep Ollama and Recallia running. Within a few minutes Recallia installs it into Ollama by itself, switches to it and rewrites the stories (the caretaker home screen shows the progress in the **Private AI** card).
3. Optional accuracy check: `npm run ft:eval -- --model recallia-remember` (compare with `--model qwen2.5:3b`).
